# Py-CoSim
EnergyPlus Python EMS-Based Co-Simulation Framework\
Author: Liang Zhang @ UArizona\
The is a work in progress. Check all the TODOs\
This is cleaned version based on `history_versions_2/example_run_clean_version_add_ML_control-mannual input_and_chatgpt_disabled_DL_occ_change_htg_to_clg_update_dir_PyCoSim.ipynb`

## Tested Versions
EnergyPlus Version: 22.2.0\
EnergyPlus API version: 0.2\
Python: 3.8.2

## Installation
Download and install the latest version of [Conda](https://www.anaconda.com/products/distribution)\
Run Anaconda Prompt as Administrator\
Create a new conda environment:\
`$ conda create -n <enter-the-name-of-repository-here> python=3.8.2 pip`
`$ activate <enter-the-name-of-repository-here>`
Ensure that you have navigated to the top level of your cloned repository. You will execute all your pip commands from this location. For example:\
`$ cd /path/to/repository`
Install the environment needed for this repository:\
`$ pip install -e .[dev]`

### Potential Usage
For single buildings
1. Test latest ML and control algorithms for single bulidings

For multiple buildings (community)
1. Community-scale co-simulation and operations among buildings, DER, ESS, and interdisciplianry components

### Connection with Other Components
1. Distributed Energy Resources (DER)
2. Energy Storage (Battery and Ice Storage)
3. Decision Making and Optimization Component
4. Artificial Intellegence, such as deep learning models (DL) and large language models (LLM)

### Paper
1. `Box\Liang and Alyssa Collaboration Folder\Python-EMS-Based Building Simulation Infrastructure`

### Resources
1. https://github.com/jmarrec/OpenStudio_to_EnergyPlusAPI/blob/main/OpenStudio_to_EnergyPlusAPI.ipynb
2. Py EMS API documentation
https://nrel.github.io/EnergyPlus/api/python/datatransfer.html
3. Available variable handles: EnergyPlusV22-2-0 : Input Output Reference: Page 2831

### TODO
1. function_n_data (bulding components)
- TODO: Inside function_n_data, eppy is used to identify all zone names and controllables
- TODO: use decorator
- TODO: export facility electricity, but don't know what is the variable handels.
- TODO: need a standardized timestamp. Should the timestamp be a global variable?
- TODO: used to use number of files to control timestep of two EnergyPlus models, next step should be using timestamp to sync all EnergyPlus models
2. Message Systems
- Use existing Python package 1) ZeroMQ, 2) RabbitMQ, and 3) pypubsub
3. Component Prioritazation
- Add attribute of prioritation for each component to decide the order of component run
4. Multiprocessing
- With threading, communication between running multiple instances is quite easy as you can just communicate using a messaging library like pypubsub to initiate signaling between the threads. With multiprocessing, communication is more difficult, but there are likely ways around it.
- mpi4py: "I am currently also working on a parallel programming project using the E+-API in Python.We are using mpi4py for the communication between the different instances of the E+-Buildings. We wanted to divide on core into multiple threads, and on each thread we could simulate one building at the same time. But in python there is the Global Interpreter lock, which usually is not allowing any multithreading applications.(https://realpython.com/python-gil/)"
5. TODOs in the code
6. Others
- Rhinoceros, Grasshopper, honeybee??Honeybee connects Rhino geometry and Grasshopper functionality to a selection of advanced energy modeling and simulation programs such as Radiance, Daysim, EnergyPlus, OpenStudio, and gbXML.
- Work with dompc: https://github.com/do-mpc/do-mpc
- Work with NeuroMANCER: https://github.com/pnnl/neuromancer
- In terms of parallel computing: another way to do it is to create multiple python script and run them together through bash: e.g., "python script1.py & python script2.py"
- Modualize
- hardcode year to 2009
- Docker
- Stability of online visualization

In [1]:
# built-in envrionment
import glob
import time
import datetime
import sys
sys.path.insert(0, 'C:/EnergyPlusV22-2-0/') # add EnergyPlus directory to system path
import os
working_path = os.getcwd() # get working path


# Py EMS modules
from pyenergyplus.api import EnergyPlusAPI

# Enable online plotting in Jupyter Notebook
%matplotlib notebook
import matplotlib.pyplot as plt

# basic data science package
import numpy as np
import pandas as pd

# Make co-simulation without randomness
# np.random.seed(123)

# threading for run components at the same time
from threading import Thread

# Enable Interactive user input
import tkinter as tk
from tkinter import simpledialog

# OpenAI to use LLM
import openai

# Pytorch for deep learning
import torch.nn as nn
import torch
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE' #enable the threading to work with PyTorch

#packages about IML
np.random.seed(42)

In [2]:
# generate random cooling setpoint
clg_random = np.random.uniform(22, 30, size=(1000,1))

### Building 1 Class

In [3]:
class function_n_data:
    
    """
    __init__: defines class inputs and variables
    init_plot: defines the initial status of online visualization
    update_line: defines how online visualization is updated
    callback_function: the key function that define the interaction between Python and EnergyPlus
    """

    def __init__(self, building_id, building_info, filename_to_run, weather_file, result_folder):
        # Class inputs
        ## Building Characteristics
        self.building_id = building_id
        self.building_info = building_info
        ## Define directories and files
        self.filename_to_run = filename_to_run
        self.weather_file = weather_file
        self.result_folder = result_folder
        
        # Class variables
        ## handle controls
        self.got_handles = False
        self.oa_temp_handle = -1
        self.zone_temp_handle = -1
        self.zone_htg_tstat_handle = -1
        self.zone_clg_tstat_handle = -1
        ## plot settings
        self.plot_update_interval = 1  # time steps
        ## Data storage
        ### Simulation output
        self.y_outdoor = []
#         self.y_radiation = []
        self.y_zone = []
#         self.y_occ = []
        self.y_htg = []
        self.y_clg = []
        self.x = []
        self.energy = []
        ### Simulation timesteps (int)
        self.count = 0
        ### Aggregated simulation and control output
        self.variables_df = pd.DataFrame([])
        self.controls_df = pd.DataFrame([])
        ### Time variables
        self.years = []
        self.months = []
        self.days = []
        self.hours = []
        self.minutes = []
        self.current_times = []
        self.actual_date_times = []
        self.actual_times = []
        
    def callback_function(self, state_argument):
        
        if not self.got_handles:
            if not api.exchange.api_data_fully_ready(state_argument):
                return

            # Variable handles
            ## Zone 1
            zone_1 = "Perimeter_ZN_1"
            self.oa_temp_handle = api.exchange.get_variable_handle(
                state_argument,u"SITE OUTDOOR AIR DRYBULB TEMPERATURE", u"ENVIRONMENT")
            self.oa_radiation_handle = api.exchange.get_variable_handle(
                state_argument,u"Site Direct Solar Radiation Rate per Area", u"ENVIRONMENT")
            self.zone_temp_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Mean Air Temperature", zone_1)
            self.zone_htg_tstat_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Thermostat Heating Setpoint Temperature", zone_1)
            self.zone_clg_tstat_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Thermostat Cooling Setpoint Temperature", zone_1)
            self.Facility_Total_Purchased_Electricity_Rate_W_handle = api.exchange.get_variable_handle(
                state_argument, "Zone Air System Sensible Cooling Rate",zone_1)
            self.zone_occupancy_handle = api.exchange.get_variable_handle(
                state_argument, "Zone People Occupant Count", zone_1)
            
            # Actuarator handles
            # Zone 1
            self.handle_clgsetp = api.exchange.get_actuator_handle(
                state_argument, "Schedule:Compact", "Schedule Value", "CLGSETP_SCH")
            self.handle_htgsetp = api.exchange.get_actuator_handle(
                state_argument, "Schedule:Compact", "Schedule Value", "HTGSETP_SCH")
            self.handle_occupancy = api.exchange.get_actuator_handle(
                state_argument, "Schedule:Compact", "Schedule Value", "BLDG_OCC_SCH")
            
            # Check whether all handels are read correctly
            if -1 in [self.oa_temp_handle, self.zone_temp_handle, self.zone_htg_tstat_handle,
                      self.zone_clg_tstat_handle, self.handle_clgsetp,  self.handle_htgsetp,
                      self.Facility_Total_Purchased_Electricity_Rate_W_handle, self.handle_occupancy, 
                      self.oa_radiation_handle, self.zone_occupancy_handle]:
                print("***Invalid handles, check spelling and sensor/actuator availability")
                sys.exit(1)
            self.got_handles = True
            
#             # draw initial plot
#             self.init_plot()

        if api.exchange.warmup_flag(state_argument):
            return
        self.count += 1
        
        # get variables via handle and save variables
        ## outdoor air temperature
        oa_temp = api.exchange.get_variable_value(state_argument, self.oa_temp_handle)
        self.y_outdoor.append(oa_temp)
        ## outdoor solar radiation
        oa_radiation = api.exchange.get_variable_value(state_argument, self.oa_radiation_handle)
#         self.y_radiation.append(oa_radiation)
        ## zone air temperature
        zone_temp = api.exchange.get_variable_value(state_argument, self.zone_temp_handle)
        self.y_zone.append(zone_temp)
        ## heating setpoint
        zone_htg_tstat = api.exchange.get_variable_value(state_argument, self.zone_htg_tstat_handle)
        self.y_htg.append(zone_htg_tstat)
        ## cooling setpoint
        zone_clg_tstat = api.exchange.get_variable_value(state_argument, self.zone_clg_tstat_handle)
        self.y_clg.append(zone_clg_tstat)
        ## Electricity rate
        Facility_Total_Purchased_Electricity_Rate_W = api.exchange.get_variable_value(
            state_argument, self.Facility_Total_Purchased_Electricity_Rate_W_handle)
        self.energy.append(Facility_Total_Purchased_Electricity_Rate_W)
        # building occupancy
        zone_occupancy = api.exchange.get_variable_value(state_argument, self.zone_occupancy_handle)
#         self.y_occ.append(zone_occupancy)

        # Read Time
        year = api.exchange.year(state)
        month = api.exchange.month(state)
        day = api.exchange.day_of_month(state)
        hour = api.exchange.hour(state)
        minute = api.exchange.minutes(state)
        current_time = api.exchange.current_time(state)
        actual_date_time = api.exchange.actual_date_time(state)
        actual_time = api.exchange.actual_time(state)
  
        # save time
        # TODO: now harcode it to 2009, Year is bogus, seems to be reading the weather file year instead...
        year = 2009
        self.years.append(year)
        self.months.append(month)
        self.days.append(day)
        self.hours.append(hour)
        self.minutes.append(minute)

        self.current_times.append(current_time)
        self.actual_date_times.append(actual_date_time)
        self.actual_times.append(actual_time)

        timedelta = datetime.timedelta()
        if hour >= 24.0:
            hour = 23.0
            timedelta += datetime.timedelta(hours=1)
        if minute >= 60.0:
            minute = 59
            timedelta += datetime.timedelta(minutes=1)

        dt = datetime.datetime(year=year, month=month, day=day, hour=hour, minute=minute)
        dt += timedelta
        self.x.append(dt)        

        # aggregate and varialbes to SMMM
        current_df = pd.DataFrame([[oa_temp, oa_radiation, zone_temp, zone_htg_tstat, zone_clg_tstat, zone_occupancy, Facility_Total_Purchased_Electricity_Rate_W]], columns = ['oa_temp', 'oa_radiation', 'zone_temp', 'zone_htg_tstat', 'zone_clg_tstat', 'zone_occ','clg_rate'])
        current_df = current_df.T
        current_df.to_csv(f'dynamic_simulation_result/{self.building_id}_{self.count:03}.csv')
        
        # get control signals calculated from control components to control the EnergyPlus models 
        ## 1.1 read control signal
        
        while True:
            try:
                clg_spt = pd.read_csv(
                    working_path + f'/dynamic_simulation_result/control_{self.count:03}.csv')['clg_spt'][0]
                htg_spt = pd.read_csv(
                    working_path + f'/dynamic_simulation_result/control_{self.count:03}.csv')['htg_spt'][0]
                break
            except:
                time.sleep(1)
                
        ## 1.2 set controls
        api.exchange.set_actuator_value(state_argument, self.handle_clgsetp, clg_spt)
        api.exchange.set_actuator_value(state_argument, self.handle_htgsetp, htg_spt)

### Building Components and Non-Builidng Components Functions
func1(), func2(): building components
func3(): TOU (static data), func4(): advanced model-based control (deep learning algorithm), func5(): Large Language Model interaction (API)

In [4]:
# Building 1
def func1():
    global api, state
    api = EnergyPlusAPI()
    state = api.state_manager.new_state()
    
    building_id = 'bldg_1'
    building_info = None #HOUSEID_1_info
    filename_to_run = "EnergyPlus_models/RefBldgSmallOfficeNew2004_v1.4_7.2_2B_USA_AZ_PHOENIX/RefBldgSmallOfficeNew2004_v1.4_7.2_2B_USA_AZ_PHOENIX_IML.idf"
    weather_file = 'epw_weather_files/USA_AZ_Tucson.Intl.AP.722740_TMY3.epw'
    result_folder = 'results/RefBldgSmallOfficeNew2004_v1.4_7.2_2B_USA_AZ_PHOENIX_IML/'

    g = function_n_data(building_id=building_id, building_info=building_info, filename_to_run=filename_to_run, weather_file=weather_file, result_folder=result_folder)

    api.runtime.callback_begin_zone_timestep_after_init_heat_balance(state, g.callback_function)
    api.runtime.run_energyplus(state,
        [
            '-w', g.weather_file,
            '-d', g.result_folder,
            g.filename_to_run
        ]
    )

    # reset the state first to call run_energyplus again
    api.state_manager.reset_state(state)

def func2():
    
    # random perturbation
    while True: 
        number_bldg_1 = 0
        
        try:
            number_bldg_1 = max([int(x[-7:-4]) for x in os.listdir(working_path + '/dynamic_simulation_result') if 'bldg_1' in x])
            
            if number_bldg_1 != 0:
#                 bldg_1 = pd.read_csv(working_path + f'/dynamic_simulation_result/bldg_1_{number_bldg_1:03}.csv')
                
                # random perturbation
                htg_spt = 20
                clg_spt = clg_random[number_bldg_1][0]

                # save the file
                spt = pd.DataFrame([clg_spt,htg_spt], index = ['clg_spt','htg_spt']).T
                spt.to_csv(working_path + f'/dynamic_simulation_result/control_{number_bldg_1:03}.csv')
        except:
            time.sleep(0.5)

### Run Co-Simulations

In [5]:
# delete all generated dynamic temp files
files = glob.glob(working_path + '/dynamic_simulation_result/*')
for f in files:
    os.remove(f)

if __name__ == '__main__':
    Thread(target = func1).start()
    Thread(target = func2).start()